# ClaimPilot Fraud Detection Model Training
Training an Isolation Forest model on synthetic insurance claims data

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE
import joblib
import os

print('All imports successful')

In [ ]:
# Load synthetic training data
data_path = '../data/fraud_training/fraud_training_data.json'
if not os.path.exists(data_path):
    from backend.mock_data.generate_fraud_training import generate_training_data
    generate_training_data(10000, 0.01)

with open(data_path, 'r') as f:
    data = json.load(f)

print(f'Loaded {data["total"]} records ({data["fraud_count"]} fraud, {data["legitimate_count"]} legitimate)')
records = data['records']

In [ ]:
# Prepare feature matrix
X = []
y = []
for r in records:
    feat = r['features']
    X.append([
        feat['days_since_policy_start'],
        feat['claim_amount'],
        feat['num_prior_claims_12mo'],
        feat['submission_delay_days'],
        feat['incident_time_of_day'],
        feat['provider_claim_frequency'],
        feat['claim_vs_policy_limit_ratio'],
    ])
    y.append(1 if r['is_fraud'] else 0)

X = np.array(X)
y = np.array(y)

feature_names = [
    'days_since_policy_start', 'claim_amount', 'num_prior_claims_12mo',
    'submission_delay_days', 'incident_time_of_day',
    'provider_claim_frequency', 'claim_vs_policy_limit_ratio'
]

print(f'X shape: {X.shape}, y distribution: {np.bincount(y)}')

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train fraud ratio: {y_train.mean():.4f}, Test fraud ratio: {y_test.mean():.4f}')

In [ ]:
# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print(f'After SMOTE - Train: {X_train_resampled.shape}, y distribution: {np.bincount(y_train_resampled)}')

In [ ]:
# Train Isolation Forest model
model = IsolationForest(
    n_estimators=200,
    max_samples='auto',
    contamination=0.01,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train_resampled)
print('Model trained successfully')

In [ ]:
# Evaluate
y_pred = model.predict(X_test)
y_scores = model.score_samples(X_test)

# Convert: Isolation Forest returns 1 for inliers, -1 for outliers
y_pred_binary = np.where(y_pred == -1, 1, 0)
# Normalize scores to 0-1 (higher = more anomalous)
y_scores_normalized = 1 - (y_scores - y_scores.min()) / (y_scores.max() - y_scores.min())

print('Classification Report:')
print(classification_report(y_test, y_pred_binary, target_names=['Legitimate', 'Fraud']))

cm = confusion_matrix(y_test, y_pred_binary)
print(f'Confusion Matrix:\n{cm}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_scores_normalized):.4f}')

In [ ]:
# Plot ROC curve
fpr, tpr, _ = roc_curve(y_test, y_scores_normalized)
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc_score(y_test, y_scores_normalized):.3f}')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Isolation Forest')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Feature importance analysis
# Isolation Forest doesn't provide direct feature importance,
# but we can analyze feature distributions for fraud vs legitimate

X_df = pd.DataFrame(X, columns=feature_names)
X_df['is_fraud'] = y

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feature in enumerate(feature_names):
    ax = axes[i]
    fraud_data = X_df[X_df['is_fraud'] == 1][feature]
    legit_data = X_df[X_df['is_fraud'] == 0][feature]
    ax.hist(legit_data, bins=30, alpha=0.5, label='Legitimate', color='green')
    ax.hist(fraud_data, bins=30, alpha=0.5, label='Fraud', color='red')
    ax.set_title(feature)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Save model
model_dir = '../backend/models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'fraud_isolation_forest.joblib')
joblib.dump(model, model_path)
print(f'Model saved to {model_path}')

# Setup instructions for ClaimPilot
print('\nModel ready for ClaimPilot integration.')
print('Place model at: backend/models/fraud_isolation_forest.joblib')